<a href="https://colab.research.google.com/github/pablonvsx/pisi3-ufrpe/blob/main/data-science/notebooks/ML/analise_final/SMOTE_COM_PESOS_MANUAIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORT DE BIBLIOTECAS
!pip install imbalanced-learn -q

from imblearn.over_sampling import SMOTE
from collections import Counter
from imblearn.pipeline import Pipeline



import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
SEED = 42

In [2]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/water_quality_2000_2008_novorotulo.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/water_quality_2000_2008.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (59896, 23)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status,Year
0,Canada,FISW_32,Lake,2003-12-02,0.043792,2.13333,9.824,0.00200,7.7900,12.00000,...,Excellent,1,1,1,1,2.0,1,5,Adequada,2003
1,Canada,IEEA_10_32,Lake,2001-06-08,0.015920,0.55000,9.824,0.00400,7.7900,16.80000,...,Excellent,1,1,1,1,2.0,1,5,Adequada,2001
2,Canada,CHRW-1876,River,2000-01-12,0.064400,10.87500,11.250,0.03590,8.2833,12.76150,...,Good,1,1,0,1,1.0,1,4,Boa,2000
3,Canada,ES063ESPFAA0000714,River,2004-01-12,1.071725,1.24444,5.850,0.20425,7.1000,18.32500,...,Fair,1,1,1,0,3.7,1,4,Boa,2004
4,Canada,CZPLA_391,River,2003-01-12,0.039740,1.83333,11.050,0.06100,7.7500,8.66667,...,Good,1,1,1,0,2.0,1,4,Boa,2003


In [3]:
X = df[[
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Country",
    "Waterbody Type",
    "Nitrogen (mg/l)"
]]

y = df["conama_status"]

In [4]:
# DIVISÃO TREINO/TESTE
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (47916, 5)
Teste: (11980, 5)


In [5]:
# PRÉ-PROCESSAMENTO
categorical_features = [
    "Country",
    "Waterbody Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [6]:
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

In [7]:
print("Antes do SMOTE:")
print(Counter(y_train))

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_encoded,
    y_train
)

print("Depois do SMOTE:")
print(Counter(y_train_smote))

Antes do SMOTE:
Counter({'Adequada': 32935, 'Boa': 9436, 'Não adequada': 5545})
Depois do SMOTE:
Counter({'Boa': 32935, 'Adequada': 32935, 'Não adequada': 32935})


In [8]:
class_weights = {
    "Adequada": 1,
    "Boa": 1,
    "Não adequada": 2
}

In [9]:
from imblearn.pipeline import Pipeline
from imblearn.combine import SMOTEENN


In [10]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        ("smote", SMOTE(random_state=SEED)),

        (
            "classifier",
            LGBMClassifier(
                random_state=SEED,
                class_weight=class_weights,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        ("balance", SMOTEENN(random_state=SEED)),

        (
            "classifier",
            LGBMClassifier(
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

In [11]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Country',
                                                   'Waterbody Type'])])),
                ('smote', SMOTE(random_state=42)),
                ('classifier',
                 LGBMClassifier(class_weight={'Adequada': 1, 'Boa': 1,
                                              'Não adequada': 2},
                                n_jobs=-1, random_state=42, verbose=-1))])

In [ ]:
y_train_pred = model.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)


Train Accuracy:
0.7012897570748811
Train Precision:
0.7689526092582828
Train Recall:
0.7012897570748811
Train F1:
0.7193793794329899

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.92      0.76      0.83     32935
         Boa       0.48      0.42      0.45      9436
Não adequada       0.37      0.83      0.51      5545

    accuracy                           0.70     47916
   macro avg       0.59      0.67      0.60     47916
weighted avg       0.77      0.70      0.72     47916

Train Confusion Matrix:
[[24985  3662  4288]
 [ 1949  3999  3488]
 [  282   644  4619]]


In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.6808848080133556

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.91      0.75      0.82      8234
         Boa       0.44      0.38      0.41      2360
Não adequada       0.34      0.76      0.47      1386

    accuracy                           0.68     11980
   macro avg       0.56      0.63      0.57     11980
weighted avg       0.75      0.68      0.70     11980


Confusion Matrix:
[[6199  896 1139]
 [ 528  898  934]
 [  90  236 1060]]


In [12]:
# com pesos manuais
y_train_pred = model.predict(X_train)

train_accuracy  = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred, average="weighted")
train_recall    = recall_score(y_train, y_train_pred, average="weighted")
train_f1        = f1_score(y_train, y_train_pred, average="weighted")
train_cm        = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:")
print(train_accuracy)

print("Train Precision:")
print(train_precision)

print("Train Recall:")
print(train_recall)

print("Train F1:")
print(train_f1)

print("\nClassification Report:")
print(classification_report(y_train, y_train_pred))

print("Train Confusion Matrix:")
print(train_cm)


Train Accuracy:
0.6890808915602304
Train Precision:
0.7834846915930274
Train Recall:
0.6890808915602304
Train F1:
0.7050523102149917

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.91      0.77      0.83     32935
         Boa       0.60      0.29      0.39      9436
Não adequada       0.32      0.90      0.47      5545

    accuracy                           0.69     47916
   macro avg       0.61      0.65      0.57     47916
weighted avg       0.78      0.69      0.71     47916

Train Confusion Matrix:
[[25243  1563  6129]
 [ 2121  2758  4557]
 [  277   251  5017]]


In [13]:
# pesos manuais
y_pred = model.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.6749582637729549

Classification Report:
              precision    recall  f1-score   support

    Adequada       0.91      0.76      0.83      8234
         Boa       0.56      0.27      0.36      2360
Não adequada       0.30      0.87      0.45      1386

    accuracy                           0.67     11980
   macro avg       0.59      0.63      0.55     11980
weighted avg       0.77      0.67      0.69     11980


Confusion Matrix:
[[6248  408 1578]
 [ 560  638 1162]
 [  94   92 1200]]
